# Export YOLO `.pt` to TensorRT `.engine`

This notebook is focused on one objective: export a trained YOLO checkpoint (`.pt`) to TensorRT (`.engine`) and quickly validate it.


## 1. What This Notebook Does

- Validates GPU/CUDA/TensorRT availability.
- Exports a YOLO `.pt` model to TensorRT `.engine`.
- Runs an optional sanity prediction.
- Includes an optional quick speed comparison (`.pt` vs `.engine`).


## 2. Environment Check

Run this first. If TensorRT is missing, install it in your environment before exporting.


In [1]:
from __future__ import annotations

from pathlib import Path
import os
import torch

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Torch version: {torch.__version__}")


Project root: /home/robiotec/Documents/Entrenamientos/Training
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
Torch version: 2.10.0+cu128


In [ ]:
# Optional: quick driver/GPU check
!nvidia-smi


In [ ]:
# TensorRT version check (required for .engine export/inference)
import tensorrt
print(f"TensorRT version: {tensorrt.__version__}")


## 3. Configure Paths and Export Settings

Edit only this cell for each export.


In [ ]:
from ultralytics import YOLO

# Input model (.pt)
PT_MODEL_PATH = PROJECT_ROOT / "result" / "Vetas" / "your_run" / "weights" / "best.pt"

# Dataset yaml is needed for some export options (especially INT8 workflows)
DATA_YAML_PATH = PROJECT_ROOT / "configs" / "datasets" / "vetas.yaml"

# Export configuration
EXPORT_CFG = {
    "format": "engine",    # TensorRT
    "imgsz": 640,
    "device": 0,
    "dynamic": True,
    "half": True,           # FP16 when supported
    "int8": False,          # Set True only if you prepared calibration flow
    "batch": 1,
    # "workspace": 8,       # Optional TensorRT workspace in GB
    "data": str(DATA_YAML_PATH),
}

print(f"PT model: {PT_MODEL_PATH}")
print(f"Data yaml: {DATA_YAML_PATH}")
print(f"Export cfg: {EXPORT_CFG}")


## 4. Export `.pt` -> `.engine`


In [ ]:
model = YOLO(str(PT_MODEL_PATH))
export_result = model.export(**EXPORT_CFG)

print("Export output:", export_result)


## 5. Resolve Engine Path

Ultralytics usually writes the engine next to the `.pt` file with the same base name.


In [ ]:
ENGINE_PATH = PT_MODEL_PATH.with_suffix(".engine")
print(f"Expected engine path: {ENGINE_PATH}")
print(f"Engine exists: {ENGINE_PATH.exists()}")


## 6. Sanity Inference (Optional)

Use a folder of test images to verify the exported engine works.


In [ ]:
TEST_SOURCE = PROJECT_ROOT / "data" / "splits" / "your_split" / "test" / "images"

# Uncomment to run inference with the exported engine
# engine_model = YOLO(str(ENGINE_PATH), task="detect")
# engine_pred = engine_model.predict(
#     source=str(TEST_SOURCE),
#     conf=0.25,
#     device=0,
#     save=True,
#     project=str(PROJECT_ROOT / "result" / "engine_checks"),
#     name="sanity_predict",
#     exist_ok=True,
# )


## 7. Quick PT vs Engine Speed Check (Optional)


In [ ]:
import time

# Uncomment to benchmark the same source with both formats
# pt_model = YOLO(str(PT_MODEL_PATH))
# engine_model = YOLO(str(ENGINE_PATH), task="detect")
#
# t0 = time.time()
# _ = pt_model.predict(source=str(TEST_SOURCE), conf=0.25, device=0, save=False, verbose=False)
# t1 = time.time()
# _ = engine_model.predict(source=str(TEST_SOURCE), conf=0.25, device=0, save=False, verbose=False)
# t2 = time.time()
#
# print(f"PT inference time: {t1 - t0:.2f}s")
# print(f"Engine inference time: {t2 - t1:.2f}s")


## 8. Notes and Troubleshooting

- If export fails, verify CUDA/TensorRT compatibility with your installed PyTorch and GPU driver.
- If `int8=True`, make sure your calibration/data setup is correct.
- If you move files after export, update `PT_MODEL_PATH` / `ENGINE_PATH` accordingly.
